# Lab 11: MapReduce Total Hours per Employee

This notebook calculates total hours worked by each employee from attendance logs
using Hadoop Streaming with Python mapper and reducer scripts.

In [1]:
%%writefile attendance_input.txt
E001,2026-04-01,09:00,17:30
E001,2026-04-02,09:15,18:00
E002,2026-04-01,08:45,17:00
E002,2026-04-02,09:00,16:30
E003,2026-04-01,10:00,19:00
E003,2026-04-02,10:15,18:45

Writing attendance_input.txt


In [2]:
%%writefile mapper_attendance.py
import sys
from datetime import datetime

time_format = "%H:%M"

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split(',')
    if len(parts) != 4:
        continue

    emp_id, _date, check_in, check_out = parts

    in_time = datetime.strptime(check_in, time_format)
    out_time = datetime.strptime(check_out, time_format)
    minutes = int((out_time - in_time).total_seconds() // 60)

    if minutes < 0:
        minutes += 24 * 60

    hours = minutes / 60.0
    print(f"{emp_id}\t{hours:.2f}")

Writing mapper_attendance.py


In [3]:
%%writefile reducer_attendance.py
import sys

current_emp = None
running_total = 0.0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    emp_id, hours = line.split("\t", 1)
    hours = float(hours)

    if current_emp is None:
        current_emp = emp_id
        running_total = hours
    elif emp_id == current_emp:
        running_total += hours
    else:
        print(f"{current_emp}\t{running_total:.2f}")
        current_emp = emp_id
        running_total = hours

if current_emp is not None:
    print(f"{current_emp}\t{running_total:.2f}")

Writing reducer_attendance.py


In [4]:
%%bash
set -e
cat attendance_input.txt | python3 mapper_attendance.py | sort | python3 reducer_attendance.py > attendance_local_output.txt
echo "Local mapper+reducer output:"
cat attendance_local_output.txt

Local mapper+reducer output:
E001	17.25
E002	15.75
E003	17.50


In [5]:
from datetime import datetime
from pathlib import Path
import time

start = time.perf_counter()
totals = {}
for line in Path("attendance_input.txt").read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue

    emp_id, _date, check_in, check_out = line.split(',')
    in_time = datetime.strptime(check_in, "%H:%M")
    out_time = datetime.strptime(check_out, "%H:%M")
    minutes = int((out_time - in_time).total_seconds() // 60)
    if minutes < 0:
        minutes += 24 * 60

    totals[emp_id] = totals.get(emp_id, 0.0) + minutes / 60.0

elapsed_ms = (time.perf_counter() - start) * 1000.0

with open("attendance_cpu_output.txt", "w", encoding="utf-8") as out:
    for emp_id in sorted(totals):
        out.write(f"{emp_id}\t{totals[emp_id]:.2f}\n")

with open("attendance_cpu_time_ms.txt", "w", encoding="utf-8") as t:
    t.write(f"{elapsed_ms:.6f}")

print(f"CPU baseline completed in {elapsed_ms:.3f} ms")

CPU baseline completed in 1.103 ms


In [6]:
%%bash
set -e

if ! command -v java >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq openjdk-11-jdk-headless
fi

JAVA_BIN="$(readlink -f "$(command -v java)")"
JAVA_HOME_DIR="$(dirname "$(dirname "$JAVA_BIN")")"

if command -v hadoop >/dev/null 2>&1; then
  HADOOP_BIN="$(command -v hadoop)"
  HADOOP_HOME="$(cd "$(dirname "$HADOOP_BIN")/.." && pwd)"
else
  HADOOP_VERSION=3.3.6
  if [ ! -d "$HOME/hadoop-$HADOOP_VERSION" ]; then
    wget -q "https://archive.apache.org/dist/hadoop/common/hadoop-$HADOOP_VERSION/hadoop-$HADOOP_VERSION.tar.gz"
    tar -xzf "hadoop-$HADOOP_VERSION.tar.gz" -C "$HOME"
  fi
  HADOOP_HOME="$HOME/hadoop-$HADOOP_VERSION"
fi

echo "export JAVA_HOME=$JAVA_HOME_DIR" > hadoop_env.sh
echo "export HADOOP_HOME=$HADOOP_HOME" >> hadoop_env.sh
echo 'export PATH=$HADOOP_HOME/bin:$PATH' >> hadoop_env.sh

source hadoop_env.sh
hadoop version | head -n 1

Hadoop 3.3.6


In [7]:
%%bash
set -e

if [ -f hadoop_env.sh ]; then
  source hadoop_env.sh
fi

if ! command -v hadoop >/dev/null 2>&1; then
  echo "Hadoop not found. Run the Hadoop setup cell first." >&2
  exit 1
fi

STREAMING_JAR=$(ls "$HADOOP_HOME"/share/hadoop/tools/lib/hadoop-streaming*.jar 2>/dev/null | head -n 1)
if [ -z "$STREAMING_JAR" ]; then
  echo "Could not find hadoop-streaming jar under $HADOOP_HOME/share/hadoop/tools/lib" >&2
  exit 1
fi

rm -rf output_attendance
LOG_FILE=attendance_hadoop.log
START_MS=$(date +%s%3N)
set +e
hadoop jar "$STREAMING_JAR" \
  -D mapreduce.framework.name=local \
  -files mapper_attendance.py,reducer_attendance.py \
  -input attendance_input.txt \
  -output output_attendance \
  -mapper "python3 mapper_attendance.py" \
  -reducer "python3 reducer_attendance.py" > "$LOG_FILE" 2>&1
STATUS=$?
set -e
END_MS=$(date +%s%3N)

if [ $STATUS -ne 0 ]; then
  echo "Hadoop attendance job failed. Last 80 log lines:" >&2
  tail -n 80 "$LOG_FILE" >&2
  exit $STATUS
fi

echo $((END_MS - START_MS)) > attendance_hadoop_time_ms.txt
cat output_attendance/part-* | sort > attendance_hadoop_output.txt
echo "Hadoop output:"
cat attendance_hadoop_output.txt

Hadoop output:
E001	17.25
E002	15.75
E003	17.50


In [8]:
from pathlib import Path

cpu_time_ms = float(Path("attendance_cpu_time_ms.txt").read_text(encoding="utf-8").strip())
hadoop_time_ms = float(Path("attendance_hadoop_time_ms.txt").read_text(encoding="utf-8").strip())

cpu_output = Path("attendance_cpu_output.txt").read_text(encoding="utf-8").strip().splitlines()
hadoop_output = Path("attendance_hadoop_output.txt").read_text(encoding="utf-8").strip().splitlines()

print(f"Output match: {cpu_output == hadoop_output}")
print(f"CPU baseline time: {cpu_time_ms:.3f} ms")
print(f"Hadoop streaming time: {hadoop_time_ms:.3f} ms")
if cpu_time_ms > 0:
    print(f"Relative overhead (Hadoop/CPU): {hadoop_time_ms / cpu_time_ms:.2f}x")
print("GPU timing is not applicable for this Hadoop Streaming lab.")

Output match: True
CPU baseline time: 1.103 ms
Hadoop streaming time: 3933.000 ms
Relative overhead (Hadoop/CPU): 3565.56x
GPU timing is not applicable for this Hadoop Streaming lab.
